# 14 · Bayesian Inversion: Priors, Sampling, and Diagnostics

*Turn regularization into probability, then audit sensitivity and computation.*

**Learning objectives**

- connect Tikhonov inversion to a linear-Gaussian posterior
- identify when sampling adds information beyond an analytic solution
- compare posterior conclusions under alternative priors
- interpret R-hat, ESS, divergences, and predictive checks together

**Prerequisites:** Chapter 08; `geodef[bayes]`  
**Estimated time:** 90–120 minutes

> **Optional dependency:** this chapter requires JAX and BlackJAX. The example
> uses a two-patch fault and short chains to teach workflow, not publication-
> quality convergence.


## Motivation

A regularized point estimate hides two questions: how uncertain are the
parameters, and how sensitive are conclusions to the prior? Linear Gaussian
problems answer the first analytically. Sampling earns its cost for positivity,
unknown hyperparameters, nonlinear geometry, and posterior shapes that a local
covariance cannot describe.


## Movement 1 — one model, reorganized probabilistically

With Gaussian noise and prior,

$$p(d\mid m)\propto e^{-\frac12(Gm-d)^TW(Gm-d)},\qquad
p(m)\propto e^{-\frac12\lambda\lVert Lm\rVert^2}.$$

Bayes' theorem makes the posterior exponent the Chapter 04 objective. The
posterior mean/MAP is the regularized estimate and its covariance is $C_m$.
ABIC compares marginal likelihood after integrating $m$ out.

Sampling becomes necessary when positivity truncates the prior, when lambda is
itself uncertain, or when nonlinear geometry makes the posterior non-Gaussian.
`SlipPosterior` samples fixed-geometry slip; `RectPosterior` and `TriPosterior`
represent geometry uncertainty, collapsing linear Gaussian slip where possible.


## A tiny positive-slip posterior

We first compute the deterministic counterpart, then sample the same two-patch
problem with positivity. Four chains are preferable in real work; two short
chains keep this lesson inside its execution budget. Production analyses use
`geodef.bayes.sample(...)`; the cell spells out BlackJAX adaptation and chain
stepping once so the origin of R-hat, ESS, and divergence flags is visible.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef
import blackjax
import jax
import jax.numpy as jnp

geodef.backend.set_backend("numpy")
rng = np.random.default_rng(0)
fault = geodef.Fault.planar(
    lat=34.0, lon=-118.0, depth=8_000.0, strike=90.0, dip=30.0,
    length=20_000.0, width=10_000.0, n_length=2, n_width=1,
)
true_slip = np.array([0.4, 1.0])
station_lon, station_lat = np.meshgrid(
    np.linspace(-118.15, -117.85, 4), np.linspace(33.90, 34.10, 3)
)
station_lon, station_lat = station_lon.ravel(), station_lat.ravel()
east, north, up = fault.displacement(
    station_lat, station_lon, slip_strike=0.0, slip_dip=true_slip
)
sigma = 0.004
gnss = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=east + rng.normal(0.0, sigma, east.size),
    north=north + rng.normal(0.0, sigma, north.size),
    up=up + rng.normal(0.0, sigma, up.size),
    sigma_east=sigma, sigma_north=sigma, sigma_up=sigma,
    name="bayes_teaching_gnss",
)
linear = geodef.solve(
    fault, datasets=gnss, components="dip", regularization="damping",
    regularization_strength=1.0, bounds=(0.0, None),
)
geodef.backend.set_backend("jax")
posterior = geodef.bayes.SlipPosterior(
    fault, gnss, components="dip", mode="fixed", positive="dip",
    regularization="damping", regularization_strength=1.0,
)
key = jax.random.PRNGKey(0)
warmup_key, chain_key = jax.random.split(key)
warmup = blackjax.window_adaptation(
    blackjax.nuts, posterior.logpdf, target_acceptance_rate=0.85
)
(adapted_state, nuts_parameters), _ = warmup.run(
    warmup_key, jnp.asarray(posterior.x0), num_steps=200
)
nuts = blackjax.nuts(posterior.logpdf, **nuts_parameters)

def run_chain(initial_position, key):
    state = nuts.init(initial_position)
    keys = jax.random.split(key, 150)

    def one_step(state, step_key):
        state, info = nuts.step(step_key, state)
        return state, (state.position, info.is_divergent)

    _, (positions, divergences) = jax.lax.scan(one_step, state, keys)
    return positions, divergences

chain_keys = jax.random.split(chain_key, 2)
initials = adapted_state.position + 0.05 * jax.random.normal(
    chain_key, (2, posterior.n_params)
)
chain_outputs = [run_chain(initials[k], chain_keys[k]) for k in range(2)]
chains = np.asarray(jnp.stack([output[0] for output in chain_outputs]))
flat_samples = chains.reshape(-1, chains.shape[-1])
divergence_count = int(sum(np.asarray(output[1]).sum() for output in chain_outputs))
rhat = np.array([
    geodef.bayes.split_rhat(chains[:, :, k]) for k in range(chains.shape[-1])
])
ess = np.array([
    geodef.bayes.effective_sample_size(chains[:, :, k])
    for k in range(chains.shape[-1])
])
slip_draws = posterior.slip_draws(flat_samples)
print("R-hat:", rhat)
print("ESS:", ess)
print("divergences:", divergence_count)


In [ ]:
lower, median, upper = np.percentile(slip_draws, [5, 50, 95], axis=0)
x = np.arange(fault.n_patches)
fig, ax = plt.subplots(figsize=(6, 3.5), constrained_layout=True)
ax.errorbar(x, median, yerr=[median - lower, upper - median], fmt="o", label="90% credible interval")
ax.plot(x, true_slip, "ks", label="truth")
ax.plot(x, linear.dip_slip, "x", label="bounded Tikhonov")
ax.set(xlabel="Patch index", ylabel="Dip slip (m)", title="Posterior and deterministic estimates")
ax.legend();


A credible interval is conditional posterior probability, not a
frequentist confidence interval and not protection against wrong geometry.
Positivity makes the interval asymmetric near zero, which a symmetric linear
standard deviation cannot represent.

## Movement 2 — diagnose before believing

R-hat compares between- and within-chain variation; values near one are
necessary. ESS discounts autocorrelation; hundreds of iterations may carry the
information of far fewer independent draws. Divergences flag unreliable
Hamiltonian trajectories and often call for reparameterization, stronger
physical priors, or more warmup—not simply more retained samples.

Prior sensitivity is part of the result: repeat with defensible alternative
smoothness, positivity, and geometry priors and tabulate changes in peak slip,
moment, and depth. Prior predictive checks reject impossible configurations
before sampling. Posterior predictive checks compare predicted data envelopes
with observed and held-out data, but Chapter 13 warns that a wrong flexible
model can still pass.


In [ ]:
prediction_draws = posterior.predict(flat_samples)
q05, q50, q95 = np.percentile(prediction_draws, [5, 50, 95], axis=0)
fig, ax = plt.subplots(figsize=(6, 3.5), constrained_layout=True)
indices = np.arange(gnss.n_obs)
ax.fill_between(indices, q05, q95, alpha=0.3, label="90% predictive band")
ax.plot(indices, gnss.obs, ".", label="observed")
ax.plot(indices, q50, "-", label="predictive median")
ax.set(xlabel="Observation row", ylabel="Displacement (m)", title="Posterior predictive check")
ax.legend();
geodef.backend.set_backend("numpy")


## Checkpoint questions

**When is regularized least squares already a complete Bayesian calculation?**
<details><summary>Answer</summary>For fixed geometry, Gaussian noise, a proper
Gaussian prior, and fixed hyperparameters, mean and covariance are analytic.</details>

**Does R-hat near one establish model adequacy?**
<details><summary>Answer</summary>No. It diagnoses chain mixing, not the physics,
likelihood, or prior.</details>

**Why rerun defensible priors?**
<details><summary>Answer</summary>To distinguish data-supported conclusions from
prior-driven ones.</details>


## Common mistakes

- Reporting one chain or one seed without convergence evidence.
- Calling a credible interval unconditional while geometry and model form are fixed.
- Treating no divergences as proof of convergence or correctness.
- Hiding prior sensitivity by showing only the preferred prior.


## Recap

- Linear-Gaussian inversion already has an analytic posterior.
- Sampling adds value for constraints, hyperparameters, and nonlinear geometry.
- Prior sensitivity, convergence, and predictive checks are all required.

Use the Bayesian and assessment guides in [workflow](../docs/workflow.md).


## Exercises

Worked solutions are in `solutions/14_bayesian_inversion_solutions.ipynb`.

1. Compare the unconstrained Gaussian posterior mean with Tikhonov.
2. Switch positivity off and compare the shallow-patch interval.
3. Halve and double the prior scale and tabulate peak slip and moment.
4. Compare one long chain with four dispersed chains.
5. Challenge: design a posterior predictive check that passes under one wrong
   geometry, then expose the error with a held-out station layout.


## Further reading

- Tarantola (2005), probabilistic inverse theory.
- Hoffman & Gelman (2014), NUTS.
- Vehtari et al. (2021), rank-normalized R-hat and effective sample size.
- Gabry et al. (2019), visualization and Bayesian workflow.
